# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muzammilsharf/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The Rule: A web page is flagged as high-risk for traffic decay if it receives a significant volume of visibility (impressions > 1000) but its actual Click-Through Rate (CTR) is severely below the median expected CTR for its current search ranking position. The risk score scales logarithmically with impression volume to prioritize high-traffic failures.

The Output Labels:
- Action Label: FLAG_FOR_REVIEW
- Reason Code: SEVERE_CTR_DEFICIT

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [1]:
import pandas as pd
import numpy as np
from huggingface_hub import hf_hub_download

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse", repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet"
)
cols = ['gsc_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position']
df = pd.read_parquet(file_path, columns=cols)
df = df[(df['gsc_data_available'] == True) & (df['gsc_impressions'] >= 1000)].copy()
df['ctr'] = df['gsc_clicks'] / df['gsc_impressions']
df['position_bucket'] = df['gsc_avg_position'].round().astype(int)

# Signal 1: CTR vs position (tied to the real CTR-fix flag from the session)
signal1 = df.groupby('position_bucket')['ctr'].agg(['median', 'count'])
print("Signal 1: CTR by position bucket")
print(signal1)
print("Verdict: CONFIRMED" if signal1['median'].is_monotonic_decreasing else "Verdict: MIXED")

# Signal 2: volume vs ctr (quick-win signal)
df['volume_bucket'] = pd.cut(df['gsc_impressions'], bins=[999, 5000, 20000, 100000, np.inf],
                              labels=['low', 'med', 'high', 'very_high'])
signal2 = df.groupby('volume_bucket', observed=True)['ctr'].agg(['median', 'count'])
print("\nSignal 2: CTR by volume bucket")
print(signal2)

/home/muzammil/flyrank-ml-internship/venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Signal 1: CTR by position bucket
                   median  count
position_bucket                 
0                0.000000    184
1                0.000916    679
2                0.003103   2501
3                0.003411   4825
4                0.002546   5118
...                   ...    ...
79               0.000000      1
88               0.000476      2
90               0.000000      1
106              0.000000      1
134              0.000000      1

[76 rows x 2 columns]
Verdict: MIXED

Signal 2: CTR by volume bucket
                 median  count
volume_bucket                 
low            0.001669  31676
med            0.001974    705
high           0.000283     38


Signal 1 verdict: MIXED. CTR generally declines as position gets worse from bucket 2 onward, but positions 0-1 show near-zero CTR despite being the best rankings, backwards from expected. Most likely explanation: these are zero-click SERP features (featured snippets, answer boxes) that hold the top position but don't generate clicks, the same failure mode flagged in the weak-picks analysis below. Not model noise, a real structural pattern in top-ranked queries.

Signal 2 verdict: OPPOSITE, low confidence. The high volume bucket shows lower median CTR than low or med, opposite of the quick-win assumption that high volume pages convert better. But high has only n=38 rows, too small to trust. Per the "N Rule": a percentage without sample size is a rumor, this verdict is not reliable evidence either way.

In [3]:
import os
import pandas as pd
import numpy as np
from huggingface_hub import hf_hub_download

print("Locating cached file...")
file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse", 
    repo_type="dataset", 
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet"
)

print("Loading exact GSC columns from the drive...")
cols_to_load = ['content_hash_id', 'gsc_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position']
df = pd.read_parquet(file_path, columns=cols_to_load)

# 1. Filter for valid data and impressions >= 1000
df_filtered = df[(df['gsc_data_available'] == True) & (df['gsc_impressions'] >= 1000)].copy()

# 2. Calculate the missing CTR column safely
df_filtered['ctr'] = df_filtered['gsc_clicks'] / df_filtered['gsc_impressions']

# 3. Round the average position to group into ranking buckets
df_filtered['position_bucket'] = df_filtered['gsc_avg_position'].round().astype(int)

# 4. Group by the position to find the baseline expected CTR
position_baselines = df_filtered.groupby('position_bucket')['ctr'].median().reset_index()
position_baselines.rename(columns={'ctr': 'expected_ctr'}, inplace=True)

# 5. Merge back and calculate the deficit score
df_scored = pd.merge(df_filtered, position_baselines, on='position_bucket', how='left')
df_scored['score'] = (df_scored['expected_ctr'] - df_scored['ctr']) * np.log1p(df_scored['gsc_impressions'])

# 6. Filter only pages with a deficit
df_scored = df_scored[df_scored['score'] > 0].copy()

# 7. Assign metadata and sort
df_scored['action_label'] = 'FLAG_FOR_REVIEW'
df_scored['reason_code'] = 'SEVERE_CTR_DEFICIT'
ranked_queue = df_scored.sort_values(by='score', ascending=False)

# 8. Export safely to the required directory
output_dir = '../outputs'
os.makedirs(output_dir, exist_ok=True)

output_cols = ['content_hash_id', 'gsc_impressions', 'ctr', 'gsc_avg_position', 'score', 'action_label', 'reason_code']
ranked_queue[output_cols].to_csv(f'{output_dir}/baseline_action_score.csv', index=False)

print("Queue written to baseline_action_score.csv successfully!")

Locating cached file...
Loading exact GSC columns from the drive...
Queue written to baseline_action_score.csv successfully!


In [4]:
top20 = ranked_queue.head(20)[['content_hash_id', 'gsc_impressions', 'ctr', 'gsc_avg_position', 'score']]
print(top20.to_string(index=False))

         content_hash_id  gsc_impressions      ctr  gsc_avg_position    score
content_34a70fea29d15f24            39003 0.000051          2.764916 0.035513
content_34a70fea29d15f24            27410 0.000000          3.129405 0.034852
content_69379902126ff53f            13726 0.000000          2.532566 0.032494
content_34a70fea29d15f24            13501 0.000074          3.220873 0.031733
content_757b1fa67827358d             9719 0.000000          3.095586 0.031316
content_bb2a9972810ddd72            11213 0.000089          3.365469 0.030972
content_757b1fa67827358d            19301 0.000000          2.261644 0.030622
content_ff8941941141101f             6538 0.000000          3.361578 0.029964
content_5bc1952fcc63ab80             6090 0.000000          2.994253 0.029722
content_8d7d99f109e19aa2            21555 0.000139          2.441151 0.029576
content_ed50f7f4237a3d02            16369 0.000367          2.519458 0.029537
content_8d7d99f109e19aa2            22321 0.000179          2.46

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*


1. **39,003 impr, pos 2.8, CTR 0.005%** — FLAG_FOR_REVIEW, SEVERE_CTR_DEFICIT, High Confidence. Massive visibility at a strong position, near-zero clicks. Wrong if: zero-click SERP feature absorbing clicks.

2. **27,410 impr, pos 3.1, CTR 0.0%** — Same page as #1, different day, confirms this isn't a one-day fluke. Wrong if: temporary carousel or ad unit on that specific date.

3. **13,726 impr, pos 2.5, CTR 0.0%** — High visibility, top-3 position, zero clicks. Wrong if: image-pack or unclickable rich result.

4. **13,501 impr, pos 3.2, CTR 0.007%** — Third occurrence of the same page as #1/#2, strong persistence signal. Wrong if: recurring but explainable SERP feature, not real decay.

5. **9,719 impr, pos 3.1, CTR 0.0%** — Good position, meaningful volume, no clicks. Wrong if: navigational query, users want a specific competitor's page.

6. **11,213 impr, pos 3.4, CTR 0.009%** — Near-zero CTR despite decent volume and position. Wrong if: paid ads monopolizing clicks that day.

7. **19,301 impr, pos 2.3, CTR 0.0%** — Same page as #5, its best position across appearances, still zero CTR. Wrong if: featured snippet answering the query directly.

8. **6,538 impr, pos 3.4, CTR 0.0%** — Lower volume, same top-4-position-zero-click pattern. Wrong if: localized query, global impressions not converting.

9. **6,090 impr, pos 3.0, CTR 0.0%** — Solid position, zero clicks. Wrong if: meta title truncated on mobile devices.

10. **21,555 impr, pos 2.4, CTR 0.014%** — High volume, strong position, essentially no clicks. Wrong if: bot/scraper traffic inflating impressions.

11. **16,369 impr, pos 2.5, CTR 0.037%** — Highest CTR in this batch, still far below expected for position 2-3. Wrong if: technical audience clicking only for specific results.

12. **22,321 impr, pos 2.5, CTR 0.018%** — Second occurrence of the same page as #10. Wrong if: seasonal dip specific to this date.

13. **4,722 impr, pos 3.2, CTR 0.0%** — Lower volume, same zero-CTR pattern. Wrong if: acronym query diluting the relevant audience.

14. **20,052 impr, pos 2.4, CTR 0.02%** — High volume, strong position, weak clicks. Wrong if: outdated year in the title tag deterring clicks.

15. **21,998 impr, pos 1.6, CTR 0.023%** — Best position in the whole top-20, still near-zero CTR, most suspicious for a zero-click feature. Wrong if: page-one answer box or knowledge panel.

16. **8,458 impr, pos 2.5, CTR 0.024%** — Mid-volume, strong position. Wrong if: part of a multi-step tutorial series users already have bookmarked.

17. **6,441 impr, pos 3.4, CTR 0.016%** — Second occurrence of the same page as #8. Wrong if: position recently improved, rolling CTR hasn't caught up yet.

18. **4,251 impr, pos 3.5, CTR 0.0%** — Lower volume, weakest position in this set, zero CTR. Wrong if: controversial topic where users trust only legacy domains.

19. **4,055 impr, pos 2.9, CTR 0.0%** — Solid position, zero clicks. Wrong if: transactional search intent, page is purely informational.

20. **11,679 impr, pos 1.8, CTR 0.0074%** — Strong position, very low CTR. Wrong if: single-day viral impression spike, not a sustained pattern.

**Note on repeats:** 4 distinct pages appear more than once in this top-20 (`content_34a70fea...` ×3, `content_757b1fa6...` ×2, `content_8d7d99f1...` ×2, `content_ff894194...` ×2), each on a different date in March. A page flagged repeatedly across days is stronger evidence of a persistent problem than a single-day anomaly, worth deduping or weighting for in a future version of this queue rather than treating each day as an independent flag.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks Analysis:
The baseline heavily flags position 2 and 3 pages with massive visibility (e.g., Row 433 with 39,003 impressions) but zero clicks. These are almost certainly "zero-click" queries, image packs, or direct snippet answers rather than genuine traffic decay. This rigid mathematical heuristic completely fails to contextualize SERP (Search Engine Results Page) features, which proves exactly why an ML model is required to weigh multiple signals simultaneously.

Leakage Check:
Confirmed. The target label trend_direction was entirely excluded from the environment. The logic strictly utilized gsc_impressions, gsc_avg_position, and calculated ctr. No future-window data or proxy labels leaked into the baseline generation.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.